# Baltic SST — Yearly Comparison

This script is part of the shared group project for the course **Open Science in Practice for Coastal Ocean Data Analysis and Visualization** Co-organized by [MAGICA](https://magica-project.eu/), [CMCC](https://www.cmcc.it/), the [University of Bologna](https://www.unibo.it/en), and [EGI](https://www.egi.eu/).

* It reads Baltic SST data from Copernicus, computes monthly means per year,
and provides interactive plots for inspection:
    1. **Spatial map** — monthly mean SST with a time slider
    2. **Seasonal cycle** — area-mean SST for each year

`version 2.1.0` `23.04.2026`

---

In [1]:
import os
import warnings
import operator
from functools import reduce

import xarray as xr
import pandas as pd
import copernicusmarine
import hvplot.xarray
import hvplot.pandas

import panel as pn
pn.extension('bokeh')

warnings.filterwarnings('ignore', category=UserWarning)
warnings.filterwarnings('ignore', category=FutureWarning)

## Fetch from Copernicus Marine & cache to Zarr

Skipped automatically if zarr file in the `CACHE_PATH` already exists.
If not, uses a Coiled cluster for fast parallel computation.

In [2]:
from dotenv import load_dotenv
import fsspec

load_dotenv(f'{os.environ["HOME"]}/dotenv/protocoast.env', override=True)

fs = fsspec.filesystem(
    's3',
    anon=False,
    skip_instance_cache=True,
    key=os.environ['AWS_ACCESS_KEY_ID'],
    secret=os.environ['AWS_SECRET_ACCESS_KEY'],
    endpoint_url=os.environ['ENDPOINT_URL'],
)

username = 'moeindst'
CACHE_S3 = f's3://{username}/baltic_sst_monthly.zarr'

In [3]:
CACHE_S3

's3://moeindst/baltic_sst_monthly.zarr'

In [4]:
os.environ['ENDPOINT_URL']

'https://rustfs.vm.fedcloud.eu'

In [5]:
%%time
try:
    cache_exists = len(fs.ls(CACHE_S3)) > 0
except (FileNotFoundError, PermissionError):
    cache_exists = False

if not cache_exists:
    import coiled

    dataset_id = 'DMI-BALTIC-SST-L4-NRT-OBS_FULL_TIME_SERIE'
    ds = copernicusmarine.open_dataset(dataset_id)
    ds_baltic = ds.sel(latitude=slice(54, 66), longitude=slice(8, 30))

    region = 'eu-central-1'
    cluster = coiled.Cluster(
        region=region,
        arm=True,
        worker_vm_types=['t4g.large'],
        n_workers=30,
        name=f'moeindst_{region}',
        wait_for_workers=True,
        compute_purchase_option='spot_with_fallback',
        software='protocoast-notebook-arm',
        workspace='esip-lab',
        timeout=360,
    )
    client = cluster.get_client()

    da_monthly_raw = (
        ds_baltic['analysed_sst']
        .resample(time='ME')
        .mean()
        .load()
    ) - 273.15
    da_monthly_raw.attrs['units'] = 'degC'

    # clear stale encoding from Copernicus source — scale_factor/add_offset/
    # _FillValue get re-applied on reload and turn all values to NaN
    da_monthly_raw.encoding.clear()
    ds_to_save = da_monthly_raw.to_dataset()
    for v in list(ds_to_save.coords):
        ds_to_save[v].encoding.clear()

    ds_to_save.to_zarr(fs.get_mapper(CACHE_S3), mode='w')
    print(f'Saved to {CACHE_S3}')
    client.close()
    cluster.close()
else:
    print(f'Cache found at {CACHE_S3} — skipping fetch.')

Cache found at s3://moeindst/baltic_sst_monthly.zarr — skipping fetch.
CPU times: user 113 ms, sys: 105 ms, total: 218 ms
Wall time: 354 ms


In [8]:
dataset_id = 'DMI-BALTIC-SST-L4-NRT-OBS_FULL_TIME_SERIE'
ds = copernicusmarine.open_dataset(dataset_id)
ds_baltic = ds.sel(latitude=slice(54, 66), longitude=slice(8, 30))

INFO - 2026-04-24T06:59:19Z - Selected dataset version: "202511"
INFO - 2026-04-24T06:59:19Z - Selected dataset part: "default"


In [9]:
ds_baltic

<xarray.Dataset> Size: 22GB
Dimensions:           (time: 1181, latitude: 601, longitude: 1100)
Coordinates:
  * latitude          (latitude) float32 2kB 54.0 54.02 54.04 ... 65.98 66.0
  * longitude         (longitude) float32 4kB 8.02 8.04 8.06 ... 29.98 30.0
  * time              (time) datetime64[ns] 9kB 2023-01-30 ... 2026-04-24
Data variables:
    analysed_sst      (time, latitude, longitude) float64 6GB dask.array<chunksize=(506, 84, 1100), meta=np.ndarray>
    analysis_error    (time, latitude, longitude) float64 6GB dask.array<chunksize=(506, 84, 1100), meta=np.ndarray>
    mask              (time, latitude, longitude) float32 3GB dask.array<chunksize=(506, 84, 1100), meta=np.ndarray>
    sea_ice_fraction  (time, latitude, longitude) float64 6GB dask.array<chunksize=(506, 84, 1100), meta=np.ndarray>
Attributes: (12/48)
    Conventions:                CF-1.4, Unidata Observation Dataset v1.0
    Metadata_Conventions:       Unidata Dataset Discovery v1.0
    acknowledgment:             Please acknowledge the use of these data with...
    cdm_data_type:              grid
    comment:                    IN NO EVENT SHALL DMI OR ITS REPRESENTATIVES ...
    creator_email:              jlh@dmi.dk
    ...                         ...
    time_coverage_end:          20251111T000000Z
    time_coverage_start:        20251110T000000Z
    title:                      DMI Sea Surface Temperature analysis
    uuid:                       a1efd321-d8bd-4e12-8c6d-fae3f51adcfb
    westernmost_longitude:      -10.0
    copernicusmarine_version:   2.3.0

## Load from Zarr

In [6]:
%%time
da_monthly = xr.open_zarr(fs.get_mapper(CACHE_S3))['analysed_sst'].load()
# da_monthly

CPU times: user 4.66 s, sys: 1.39 s, total: 6.05 s
Wall time: 7.2 s


In [7]:
ds

<xarray.DataArray 'analysed_sst' (time: 40, latitude: 601, longitude: 1100)> Size: 212MB
array([[[5.82499377, 5.75999377, 5.69499377, ...,        nan,
                nan,        nan],
        [5.83999377, 5.77999377, 5.70999377, ...,        nan,
                nan,        nan],
        [5.84499377, 5.77999377, 5.71499377, ...,        nan,
                nan,        nan],
        ...,
        [7.67499372, 7.65999373, 7.63999373, ...,        nan,
                nan,        nan],
        [7.66499373, 7.64999373, 7.62999373, ...,        nan,
                nan,        nan],
        [7.64999373, 7.62999373, 7.61499373, ...,        nan,
                nan,        nan]],

       [[5.17999378, 5.16499378, 5.14606521, ...,        nan,
                nan,        nan],
        [5.17642235, 5.16070807, 5.14606521, ...,        nan,
                nan,        nan],
        [5.16999378, 5.15320807, 5.13749378, ...,        nan,
                nan,        nan],
...
        [7.60967115, 7.60902598, 7.60902598, ...,        nan,
                nan,        nan],
        [7.60289695, 7.60225179, 7.60321953, ...,        nan,
                nan,        nan],
        [7.5954776 , 7.5954776 , 7.59676792, ...,        nan,
                nan,        nan]],

       [[7.01955896, 7.05564591, 7.09477635, ...,        nan,
                nan,        nan],
        [6.98564591, 7.02042852, 7.05999374, ...,        nan,
                nan,        nan],
        [6.95390678, 6.99042852, 7.0308633 , ...,        nan,
                nan,        nan],
        ...,
        [7.93955894, 7.93955894, 7.93695024, ...,        nan,
                nan,        nan],
        [7.93651546, 7.93477633, 7.9330372 , ...,        nan,
                nan,        nan],
        [7.93347198, 7.93216763, 7.92955894, ...,        nan,
                nan,        nan]]], shape=(40, 601, 1100))
Coordinates:
  * latitude   (latitude) float32 2kB 54.0 54.02 54.04 ... 65.96 65.98 66.0
  * longitude  (longitude) float32 4kB 8.02 8.04 8.06 8.08 ... 29.96 29.98 30.0
  * time       (time) datetime64[ns] 320B 2023-01-31 2023-02-28 ... 2026-04-30
Attributes:
    units:    degC

## Split by year & build seasonal DataFrame

In [ ]:
%%time
years = sorted({int(t) for t in da_monthly.time.dt.year.values})
da_by_year = {yr: da_monthly.sel(time=str(yr)) for yr in years}

records = {}
for yr, da in da_by_year.items():
    area_mean = da.mean(dim=['latitude', 'longitude']).values
    months = da.time.dt.month.values
    records[str(yr)] = pd.Series(area_mean, index=months)

df_seasonal = pd.DataFrame(records)
df_seasonal.index.name = 'month'

In [ ]:
df_seasonal

## Plot 1 — Spatial monthly SST map with time slider

In [ ]:
import cmocean

In [ ]:
%%time
map_opts = dict(
    x='longitude', y='latitude',
    rasterize=True, geo=True,
    cmap=cmocean.cm.thermal, clim=(0, 20),
    tiles='OSM',
    width=700, height=500, alpha=0.95
)

plot1 = da_monthly.hvplot(
    title='Baltic SST Monthly Mean (degC)',
    groupby='time',
    **map_opts,
)
plot1

## Plot 2 — Seasonal cycle by year

In [ ]:
# COLORS = ['#1f77b4', '#ff7f0e', '#2ca02c', '#d62728', '#9467bd']
COLORS = ['royalblue', 'seagreen', 'orange', 'red']

curves = [
    df_seasonal[col].hvplot.line(
        label=col,
        color=COLORS[i % len(COLORS)],
        line_width=2,
    )
    for i, col in enumerate(df_seasonal.columns)
]

seasonal_plot = reduce(operator.mul, curves).opts(
    title='Baltic SST — Monthly spatial mean',
    xlabel='Month', ylabel='SST (degC)',
    legend_position='top_left',
    width=700, height=400,
    show_grid=True,
)
seasonal_plot

## Panel Dashboard

Serve interactively with:
```bash
panel serve baltic_sst_yearly_comparison_v2.ipynb --show
```
Opens at `http://localhost:5006`

In [ ]:
pn.Column(                                                                                                 
      pn.pane.Markdown('## Spatial Monthly Mean SST'),                                                       
      pn.panel(plot1),                                                                                       
      pn.layout.Divider(),                                                                                   
      pn.pane.Markdown('## Seasonal Cycle — Area-Mean SST by Year'),                                         
      pn.panel(seasonal_plot),                                                                               
  )

                                        ----------- End of Document -----------